# GEC-RAG Project
## Notebook 05 — LLM Baselines (GPT-4o & Llama 3.1 8B)

This notebook runs two LLM baselines for our benchmark:

1. **GPT-4o-mini (no RAG)** — State-of-the-art LLM with generic GEC prompt, no KB grounding
2. **Llama 3.1 8B (no RAG)** — Open-source LLM with the same generic prompt

**Role in the paper:**
These baselines isolate the contribution of RAG grounding. Comparing these systems
against our GEC-RAG system (Notebook 06) directly measures whether corpus-grounded
pedagogical explanations improve over generic LLM explanations.

**Llama model path:** `C:/gecrag/models/Llama-3.1-8B-Instruct`

**What this notebook produces:**
- Corrections and generic explanations for both LLMs
- F0.5 scores via ERRANT
- Per error-type performance breakdown
- Saved predictions for Notebook 07 (evaluation)

---
### Cell 1 — Imports & Setup

In [1]:
import os
import json
import time
import torch
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
from openai import OpenAI
from dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')

# ⚠️ Update BASE_DIR to match your actual path
BASE_DIR           = 'C:/gecrag'
DATA_DIR           = os.path.join(BASE_DIR, 'data/write_and_improve')
FINAL_VERSIONS_DIR = os.path.join(DATA_DIR, 'user-prompt-final-versions')
RESULTS_DIR        = os.path.join(BASE_DIR, 'results')
LLAMA_MODEL_PATH   = os.path.join(BASE_DIR, 'models/Llama-3.1-8B-Instruct')
os.makedirs(RESULTS_DIR, exist_ok=True)

# Load API key
load_dotenv(os.path.join(BASE_DIR, '.env'))
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

# Check GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✅ Device  : {device}')
if device == 'cuda':
    print(f'✅ GPU     : {torch.cuda.get_device_name(0)}')
print(f'✅ Torch   : {torch.__version__}')

# Quick API test
test = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{'role': 'user', 'content': 'Say OK'}],
    max_tokens=5
)
print(f'✅ OpenAI  : {test.choices[0].message.content}')

✅ Device  : cuda
✅ GPU     : NVIDIA GeForce RTX 4070
✅ Torch   : 2.5.1+cu121
✅ OpenAI  : OK!


---
### Cell 2 — Verify Llama Model Files

In [2]:
# Verify all required Llama files are present before loading
required_files = [
    'config.json',
    'tokenizer.json',
    'tokenizer_config.json',
    'special_tokens_map.json',
    'model-00001-of-00004.safetensors',
    'model-00002-of-00004.safetensors',
    'model-00003-of-00004.safetensors',
    'model-00004-of-00004.safetensors',
    'model.safetensors.index.json',
]

print(f'Checking Llama model files at:')
print(f'  {LLAMA_MODEL_PATH}\n')

all_present = True
total_size  = 0
for fname in required_files:
    fpath = os.path.join(LLAMA_MODEL_PATH, fname)
    if os.path.exists(fpath):
        size_gb = os.path.getsize(fpath) / (1024**3)
        total_size += os.path.getsize(fpath)
        print(f'  ✅ {fname:<45} ({size_gb:.2f} GB)')
    else:
        print(f'  ❌ {fname:<45} MISSING')
        all_present = False

print(f'\n  Total size : {total_size/(1024**3):.2f} GB')
if all_present:
    print(f'  ✅ All files present — ready to load')
else:
    print(f'  ❌ Some files missing — check download')

Checking Llama model files at:
  C:/gecrag\models/Llama-3.1-8B-Instruct

  ✅ config.json                                   (0.00 GB)
  ✅ tokenizer.json                                (0.01 GB)
  ✅ tokenizer_config.json                         (0.00 GB)
  ✅ special_tokens_map.json                       (0.00 GB)
  ✅ model-00001-of-00004.safetensors              (4.63 GB)
  ✅ model-00002-of-00004.safetensors              (4.66 GB)
  ✅ model-00003-of-00004.safetensors              (4.58 GB)
  ✅ model-00004-of-00004.safetensors              (1.09 GB)
  ✅ model.safetensors.index.json                  (0.00 GB)

  Total size : 14.97 GB
  ✅ All files present — ready to load


---
### Cell 3 — Load Dev Set

In [3]:
def load_text_file(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        return [line.strip() for line in f if line.strip()]

orig_path = os.path.join(
    FINAL_VERSIONS_DIR,
    'en-writeandimprove2024-final-versions-dev-sentences.orig'
)
corr_path = os.path.join(
    FINAL_VERSIONS_DIR,
    'en-writeandimprove2024-final-versions-dev-sentences.corr'
)

orig_sentences = load_text_file(orig_path)
corr_sentences = load_text_file(corr_path)
assert len(orig_sentences) == len(corr_sentences)

print(f'✅ Dev split loaded: {len(orig_sentences)} sentence pairs')
print(f'\nSample:')
for i in range(3):
    print(f'  [{i+1}] ORIG: {orig_sentences[i]}')
    print(f'       CORR: {corr_sentences[i]}')

✅ Dev split loaded: 4352 sentence pairs

Sample:
  [1] ORIG: There are lots of important things in our life.
       CORR: There are lots of important things in our life.
  [2] ORIG: However, in most cases, they depend on the priorities we have and acceptation we are awaiting from it.
       CORR: However, in most cases, they depend on the priorities we have and the acceptation we are awaiting from it.
  [3] ORIG: If it fulfils our dreams and aims or not?
       CORR: If it fulfils our dreams and aims or not.


---
### Cell 4 — Define Shared GEC Prompt
Both LLMs receive the same prompt — generic GEC with no KB grounding.
This is the key comparison point against our RAG system in Notebook 06.

In [4]:
SYSTEM_PROMPT = """You are a grammatical error correction assistant.
Your task is to correct grammatical errors in English sentences written by L2 learners.

For each sentence you receive, respond with a JSON object containing:
- "corrected": the corrected sentence (if no errors, return the original sentence unchanged)
- "explanation": a brief explanation of what was corrected and why (1-3 sentences).
  If no correction was needed, set explanation to null.

Respond ONLY with the JSON object. No preamble or extra text."""

def build_user_prompt(sentence):
    return f'Correct this sentence:\n\n{sentence}'

# Quick GPT-4o test
test_response = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': build_user_prompt('I go to school yesterday .')}
    ],
    max_tokens=200,
    temperature=0
)
print('✅ Shared prompt defined')
print(f'\nGPT-4o test response:')
print(test_response.choices[0].message.content)

✅ Shared prompt defined

GPT-4o test response:
{
  "corrected": "I went to school yesterday.",
  "explanation": "The verb 'go' was changed to 'went' to correctly reflect the past tense, as indicated by the use of 'yesterday'."
}


---
### Cell 5 — GPT-4o Inference
> ⚠️ Set MAX_SENTENCES=50 for a quick test first.
> ⏳ Full run (~4352 sentences) costs approximately $3-5 in API credits.

In [13]:
def run_gpt4o(sentences, client, system_prompt, model='gpt-4o-mini', max_sentences=None):
    results   = []
    errors    = 0
    n         = max_sentences or len(sentences)
    sentences = sentences[:n]

    for sent in tqdm(sentences, desc=f'GPT-4o ({model})'):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {'role': 'system', 'content': system_prompt},
                    {'role': 'user',   'content': build_user_prompt(sent)}
                ],
                max_tokens=300,
                temperature=0
            )
            raw = response.choices[0].message.content.strip()
            try:
                clean  = raw.replace('```json','').replace('```','').strip()
                parsed = json.loads(clean)
                results.append({
                    'original'   : sent,
                    'corrected'  : parsed.get('corrected', sent),
                    'explanation': parsed.get('explanation', None),
                    'raw'        : raw
                })
            except json.JSONDecodeError:
                results.append({
                    'original': sent, 'corrected': sent,
                    'explanation': None, 'raw': raw
                })
                errors += 1
            time.sleep(0.05)
        except Exception as e:
            results.append({
                'original': sent, 'corrected': sent,
                'explanation': None, 'raw': str(e)
            })
            errors += 1
            time.sleep(1)

    print(f'\n✅ GPT-4o complete | Processed: {len(results)} | Parse errors: {errors}')
    return results


# ⚠️ Change to None for full run
MAX_SENTENCES_GPT = None

print(f'Running GPT-4o on {MAX_SENTENCES_GPT or len(orig_sentences)} sentences...')
gpt4o_results = run_gpt4o(
    orig_sentences, client, SYSTEM_PROMPT,
    model='gpt-4o-mini',
    max_sentences=MAX_SENTENCES_GPT
)

print('\nSample GPT-4o results:')
for r in gpt4o_results[:3]:
    print(f'\n  ORIG : {r["original"]}')
    print(f'  CORR : {r["corrected"]}')
    print(f'  EXPL : {r["explanation"]}')

Running GPT-4o on 4352 sentences...


GPT-4o (gpt-4o-mini):   0%|          | 0/4352 [00:00<?, ?it/s]


✅ GPT-4o complete | Processed: 4352 | Parse errors: 0

Sample GPT-4o results:

  ORIG : There are lots of important things in our life.
  CORR : There are lots of important things in our lives.
  EXPL : The word 'life' should be pluralized to 'lives' to correctly refer to the multiple lives of people, as the sentence implies a general statement about many individuals.

  ORIG : However, in most cases, they depend on the priorities we have and acceptation we are awaiting from it.
  CORR : However, in most cases, they depend on the priorities we have and the acceptance we are awaiting from it.
  EXPL : The word 'acceptation' is incorrect in this context; the correct term is 'acceptance.' Additionally, 'the' was added before 'acceptance' for grammatical correctness.

  ORIG : If it fulfils our dreams and aims or not?
  CORR : Does it fulfill our dreams and aims or not?
  EXPL : The original sentence was a statement but structured as a question. It was corrected to use 'Does' at the begin

In [14]:
import gc
import torch

# Clear GPU memory from previous models (T5-GEC etc.)
if 'model' in dir():
    del model
if 'tokenizer' in dir():
    del tokenizer

gc.collect()
torch.cuda.empty_cache()

# Check available VRAM
total = torch.cuda.get_device_properties(0).total_memory / (1024**3)
reserved = torch.cuda.memory_reserved(0) / (1024**3)
free = total - reserved

print(f'GPU Memory:')
print(f'  Total    : {total:.2f} GB')
print(f'  Reserved : {reserved:.2f} GB')
print(f'  Free     : {free:.2f} GB')
print(f'\nNeed ~6GB free for Llama 4-bit — {"✅ OK" if free > 6 else "❌ Not enough"}')

GPU Memory:
  Total    : 11.99 GB
  Reserved : 5.40 GB
  Free     : 6.59 GB

Need ~6GB free for Llama 4-bit — ✅ OK


---
### Cell 6 — Load Llama 3.1 8B
> ⏳ Loading takes ~1-2 minutes. Uses 4-bit quantization to fit on RTX 4070 12GB.

In [7]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True
)

print(f'Loading Llama 3.1 8B from local path...')
print(f'  {LLAMA_MODEL_PATH}\n')

llama_tokenizer = AutoTokenizer.from_pretrained(
    LLAMA_MODEL_PATH,
    local_files_only=True
)
llama_tokenizer.pad_token = llama_tokenizer.eos_token

llama_model = AutoModelForCausalLM.from_pretrained(
    LLAMA_MODEL_PATH,
    quantization_config=bnb_config,
    device_map={'': 0},        # Force everything to GPU 0
    local_files_only=True
)
llama_model.eval()

print(f'✅ Llama 3.1 8B loaded')
print(f'   VRAM used: {torch.cuda.memory_reserved(0)/(1024**3):.2f} GB')

# Sanity check
test_msgs = [
    {'role': 'system', 'content': SYSTEM_PROMPT},
    {'role': 'user',   'content': build_user_prompt('I go to school yesterday .')}
]
test_input = llama_tokenizer.apply_chat_template(
    test_msgs, tokenize=False, add_generation_prompt=True
)
inputs = llama_tokenizer(test_input, return_tensors='pt').to('cuda')
with torch.no_grad():
    output = llama_model.generate(
        **inputs, max_new_tokens=200,
        temperature=0.01, do_sample=True,
        pad_token_id=llama_tokenizer.eos_token_id
    )
decoded = llama_tokenizer.decode(
    output[0][inputs['input_ids'].shape[1]:],
    skip_special_tokens=True
).strip()

print(f'\n🔍 Sanity check:')
print(f'   Input : I go to school yesterday .')
print(f'   Output: {decoded[:300]}')

Loading Llama 3.1 8B from local path...
  C:/gecrag\models/Llama-3.1-8B-Instruct



Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

✅ Llama 3.1 8B loaded
   VRAM used: 5.40 GB

🔍 Sanity check:
   Input : I go to school yesterday .
   Output: {
  "corrected": "I went to school yesterday.",
  "explanation": "The verb tense in the sentence should be past tense because the action of going to school happened in the past. The correct verb form is 'went' instead of 'go'."
}


---
### Cell 7 — Llama 3.1 Inference
> ⚠️ Set MAX_SENTENCES=50 for quick test first.
> ⏳ Full run takes 30-60 minutes on RTX 4070.

In [15]:
def run_llama(sentences, model, tokenizer, system_prompt, device, max_sentences=None):
    results   = []
    errors    = 0
    n         = max_sentences or len(sentences)
    sentences = sentences[:n]

    for sent in tqdm(sentences, desc='Llama 3.1 inference'):
        try:
            messages = [
                {'role': 'system', 'content': system_prompt},
                {'role': 'user',   'content': build_user_prompt(sent)}
            ]
            prompt = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
            inputs = tokenizer(
                prompt, return_tensors='pt',
                truncation=True, max_length=512
            ).to(device)

            with torch.no_grad():
                output = model.generate(
                    **inputs, max_new_tokens=300,
                    temperature=0.01, do_sample=True,
                    pad_token_id=tokenizer.eos_token_id
                )
            decoded = tokenizer.decode(
                output[0][inputs['input_ids'].shape[1]:],
                skip_special_tokens=True
            ).strip()

            try:
                clean  = decoded.replace('```json','').replace('```','').strip()
                parsed = json.loads(clean)
                results.append({
                    'original'   : sent,
                    'corrected'  : parsed.get('corrected', sent),
                    'explanation': parsed.get('explanation', None),
                    'raw'        : decoded
                })
            except json.JSONDecodeError:
                results.append({
                    'original': sent, 'corrected': sent,
                    'explanation': None, 'raw': decoded
                })
                errors += 1

        except Exception as e:
            results.append({
                'original': sent, 'corrected': sent,
                'explanation': None, 'raw': str(e)
            })
            errors += 1

    print(f'\n✅ Llama complete | Processed: {len(results)} | Parse errors: {errors}')
    return results


# ⚠️ Change to None for full run
MAX_SENTENCES_LLAMA = None

print(f'Running Llama 3.1 on {MAX_SENTENCES_LLAMA or len(orig_sentences)} sentences...')
llama_results = run_llama(
    orig_sentences, llama_model, llama_tokenizer,
    SYSTEM_PROMPT, device,
    max_sentences=MAX_SENTENCES_LLAMA
)

print('\nSample Llama results:')
for r in llama_results[:3]:
    print(f'\n  ORIG : {r["original"]}')
    print(f'  CORR : {r["corrected"]}')
    print(f'  EXPL : {r["explanation"]}')

Running Llama 3.1 on 4352 sentences...


Llama 3.1 inference:   0%|          | 0/4352 [00:00<?, ?it/s]


✅ Llama complete | Processed: 4352 | Parse errors: 7

Sample Llama results:

  ORIG : There are lots of important things in our life.
  CORR : There are many important things in our lives.
  EXPL : The original sentence uses 'lots', which is an informal or colloquial expression. In formal writing, it's more common to use'many' to express a large quantity. Additionally, the subject-verb agreement requires a plural verb 'are' to match the plural subject 'things', and the correct phrase is 'lives' instead of 'life' to indicate multiple aspects of life.

  ORIG : However, in most cases, they depend on the priorities we have and acceptation we are awaiting from it.
  CORR : However, in most cases, they depend on the priorities we have and the acceptance we are awaiting from it.
  EXPL : The word 'acceptation' is not a valid word in English. The correct word is 'acceptance'.

  ORIG : If it fulfils our dreams and aims or not?
  CORR : If it fulfils our dreams and aims or not?
  EXPL : None


---
### Cell 8 — Compute F0.5 for Both Systems

In [16]:
import errant

annotator = errant.load('en')
print('✅ ERRANT loaded')

def compute_f05(orig_list, pred_list, ref_list, annotator):
    tp = fp = fn = 0
    for orig, pred, ref in tqdm(
        zip(orig_list, pred_list, ref_list),
        total=len(orig_list), desc='Computing F0.5'
    ):
        try:
            orig_p = annotator.parse(orig)
            pred_p = annotator.parse(pred)
            ref_p  = annotator.parse(ref)
            pred_edits = set(
                (e.o_start, e.o_end, e.c_str)
                for e in annotator.annotate(orig_p, pred_p) if e.type != 'noop'
            )
            ref_edits = set(
                (e.o_start, e.o_end, e.c_str)
                for e in annotator.annotate(orig_p, ref_p) if e.type != 'noop'
            )
            tp += len(pred_edits & ref_edits)
            fp += len(pred_edits - ref_edits)
            fn += len(ref_edits  - pred_edits)
        except Exception:
            continue
    beta      = 0.5
    precision = tp/(tp+fp) if (tp+fp) > 0 else 0
    recall    = tp/(tp+fn) if (tp+fn) > 0 else 0
    f05       = ((1+beta**2)*precision*recall /
                 (beta**2*precision+recall)) if (precision+recall) > 0 else 0
    return {
        'precision': round(precision*100, 2),
        'recall'   : round(recall*100,    2),
        'f05'      : round(f05*100,       2),
        'tp': tp, 'fp': fp, 'fn': fn
    }


n_gpt   = len(gpt4o_results)
n_llama = len(llama_results)

gpt4o_preds = [r['corrected'] for r in gpt4o_results]
llama_preds = [r['corrected'] for r in llama_results]

print('Computing F0.5 for GPT-4o...')
gpt4o_scores = compute_f05(
    orig_sentences[:n_gpt], gpt4o_preds,
    corr_sentences[:n_gpt], annotator
)

print('Computing F0.5 for Llama 3.1...')
llama_scores = compute_f05(
    orig_sentences[:n_llama], llama_preds,
    corr_sentences[:n_llama], annotator
)

sep = '='*45
print(f'\n{sep}')
print('LLM Baseline Results (no RAG)')
print(sep)
print(f'\nGPT-4o-mini:')
print(f'  Precision : {gpt4o_scores["precision"]}%')
print(f'  Recall    : {gpt4o_scores["recall"]}%')
print(f'  F0.5      : {gpt4o_scores["f05"]}%')
print(f'\nLlama 3.1 8B:')
print(f'  Precision : {llama_scores["precision"]}%')
print(f'  Recall    : {llama_scores["recall"]}%')
print(f'  F0.5      : {llama_scores["f05"]}%')
print(sep)

✅ ERRANT loaded
Computing F0.5 for GPT-4o...


Computing F0.5:   0%|          | 0/4352 [00:00<?, ?it/s]

Computing F0.5 for Llama 3.1...


Computing F0.5:   0%|          | 0/4352 [00:00<?, ?it/s]


LLM Baseline Results (no RAG)

GPT-4o-mini:
  Precision : 37.4%
  Recall    : 53.73%
  F0.5      : 39.82%

Llama 3.1 8B:
  Precision : 31.17%
  Recall    : 43.79%
  F0.5      : 33.08%


---
### Cell 9 — Per Error-Type Breakdown

In [17]:
def compute_per_type_scores(orig_list, pred_list, ref_list, annotator):
    type_stats = {}
    for orig, pred, ref in tqdm(
        zip(orig_list, pred_list, ref_list),
        total=len(orig_list), desc='Per-type analysis'
    ):
        try:
            orig_p = annotator.parse(orig)
            pred_p = annotator.parse(pred)
            ref_p  = annotator.parse(ref)
            pred_edits = {
                (e.o_start, e.o_end, e.c_str): e.type
                for e in annotator.annotate(orig_p, pred_p) if e.type != 'noop'
            }
            ref_edits = {
                (e.o_start, e.o_end, e.c_str): e.type
                for e in annotator.annotate(orig_p, ref_p) if e.type != 'noop'
            }
            for key, etype in ref_edits.items():
                if etype not in type_stats:
                    type_stats[etype] = {'tp':0,'fp':0,'fn':0}
                if key in pred_edits:
                    type_stats[etype]['tp'] += 1
                else:
                    type_stats[etype]['fn'] += 1
            for key, etype in pred_edits.items():
                if key not in ref_edits:
                    if etype not in type_stats:
                        type_stats[etype] = {'tp':0,'fp':0,'fn':0}
                    type_stats[etype]['fp'] += 1
        except Exception:
            continue

    beta = 0.5
    results = []
    for etype, s in type_stats.items():
        p   = s['tp']/(s['tp']+s['fp']) if (s['tp']+s['fp']) > 0 else 0
        r   = s['tp']/(s['tp']+s['fn']) if (s['tp']+s['fn']) > 0 else 0
        f05 = ((1+beta**2)*p*r/(beta**2*p+r)) if (p+r) > 0 else 0
        results.append({
            'error_type': etype,
            'tp': s['tp'], 'fp': s['fp'], 'fn': s['fn'],
            'precision' : round(p*100, 2),
            'recall'    : round(r*100, 2),
            'f05'       : round(f05*100, 2),
        })
    return pd.DataFrame(results).sort_values('f05', ascending=False)


print('Computing per-type scores...')
gpt4o_per_type = compute_per_type_scores(
    orig_sentences[:n_gpt], gpt4o_preds,
    corr_sentences[:n_gpt], annotator
)
llama_per_type = compute_per_type_scores(
    orig_sentences[:n_llama], llama_preds,
    corr_sentences[:n_llama], annotator
)

specific_gpt   = gpt4o_per_type[~gpt4o_per_type['error_type'].str.contains('OTHER')]
specific_llama = llama_per_type[~llama_per_type['error_type'].str.contains('OTHER')]

print('\nGPT-4o Per Error-Type F0.5 (top 10):')
print(specific_gpt.head(10).to_string(index=False))
print('\nLlama 3.1 Per Error-Type F0.5 (top 10):')
print(specific_llama.head(10).to_string(index=False))

Computing per-type scores...


Per-type analysis:   0%|          | 0/4352 [00:00<?, ?it/s]

Per-type analysis:   0%|          | 0/4352 [00:00<?, ?it/s]


GPT-4o Per Error-Type F0.5 (top 10):
 error_type  tp  fp  fn  precision  recall   f05
R:NOUN:INFL   2   0   1     100.00   66.67 90.91
R:NOUN:POSS   1   0   1     100.00   50.00 83.33
    R:SPELL 358 118  78      75.21   82.11 76.50
 R:VERB:SVA 154  81  27      65.53   85.08 68.69
M:VERB:FORM  56  30  10      65.12   84.85 68.29
R:VERB:FORM 210 151 110      58.17   65.62 59.52
     R:ORTH 415 314 301      56.93   57.96 57.13
      M:DET 212 177 155      54.50   57.77 55.12
     R:PART  36  35  26      50.70   58.06 52.02
 R:NOUN:NUM 141 146  80      49.13   63.80 51.50

Llama 3.1 Per Error-Type F0.5 (top 10):
 error_type  tp  fp  fn  precision  recall   f05
 R:ADJ:FORM   4   1   3      80.00   57.14 74.07
    R:SPELL 325 130 111      71.43   74.54 72.03
R:NOUN:INFL   1   0   2     100.00   33.33 71.43
 R:VERB:SVA 137  68  44      66.83   75.69 68.43
M:VERB:FORM  47  31  19      60.26   71.21 62.17
     R:ORTH 377 274 339      57.91   52.65 56.78
R:VERB:FORM 185 157 135      54.09   57

---
### Cell 10 — Save Results

In [18]:
# Save GPT-4o predictions
gpt4o_path = os.path.join(RESULTS_DIR, 'gpt4o_predictions.json')
with open(gpt4o_path, 'w', encoding='utf-8') as f:
    json.dump({
        'system'     : 'GPT-4o-mini (no RAG)',
        'model'      : 'gpt-4o-mini',
        'n_sentences': len(gpt4o_results),
        'scores'     : gpt4o_scores,
        'predictions': gpt4o_results
    }, f, ensure_ascii=False, indent=2)
print(f'✅ GPT-4o predictions  : {gpt4o_path}')

# Save Llama predictions
llama_path = os.path.join(RESULTS_DIR, 'llama_predictions.json')
with open(llama_path, 'w', encoding='utf-8') as f:
    json.dump({
        'system'     : 'Llama 3.1 8B (no RAG)',
        'model'      : 'Llama-3.1-8B-Instruct',
        'n_sentences': len(llama_results),
        'scores'     : llama_scores,
        'predictions': llama_results
    }, f, ensure_ascii=False, indent=2)
print(f'✅ Llama predictions   : {llama_path}')

# Save per-type scores
gpt4o_per_type.to_csv(
    os.path.join(RESULTS_DIR, 'gpt4o_per_type_scores.csv'), index=False
)
llama_per_type.to_csv(
    os.path.join(RESULTS_DIR, 'llama_per_type_scores.csv'), index=False
)
print(f'✅ Per-type scores saved')

# Update shared baseline scores
scores_path = os.path.join(RESULTS_DIR, 'baseline_scores.json')
try:
    with open(scores_path, 'r') as f:
        all_scores = json.load(f)
except FileNotFoundError:
    all_scores = {}

all_scores['GPT-4o-mini (no RAG)']  = gpt4o_scores
all_scores['Llama-3.1-8B (no RAG)'] = llama_scores
with open(scores_path, 'w') as f:
    json.dump(all_scores, f, indent=2)
print(f'✅ Baseline scores     : {scores_path}')

✅ GPT-4o predictions  : C:/gecrag\results\gpt4o_predictions.json
✅ Llama predictions   : C:/gecrag\results\llama_predictions.json
✅ Per-type scores saved
✅ Baseline scores     : C:/gecrag\results\baseline_scores.json


---
### Cell 11 — Full Baseline Comparison

In [19]:
scores_path = os.path.join(RESULTS_DIR, 'baseline_scores.json')
with open(scores_path, 'r') as f:
    all_scores = json.load(f)

sep = '='*60
print(sep)
print('BASELINE COMPARISON — CORRECTION QUALITY (F0.5)')
print(sep)
print(f'\n  {"System":<30} {"Precision":>10} {"Recall":>8} {"F0.5":>8}')
print(f'  {"-"*58}')
for system, scores in all_scores.items():
    print(
        f'  {system:<30} '
        f'{scores["precision"]:>9}% '
        f'{scores["recall"]:>7}% '
        f'{scores["f05"]:>7}%'
    )

print(f'\n⚠️  Reminder: If you ran test runs (50 sentences),')
print(f'   set MAX_SENTENCES_GPT=None and MAX_SENTENCES_LLAMA=None')
print(f'   in Cells 5 and 7, then re-run cells 5-10 for full results.')

print(f'\n📁 Output Files:')
print(f'   - results/gpt4o_predictions.json')
print(f'   - results/llama_predictions.json')
print(f'   - results/gpt4o_per_type_scores.csv')
print(f'   - results/llama_per_type_scores.csv')
print(f'   - results/baseline_scores.json')

print(f'\n➡️  Next: Run Notebook 06 — GEC-RAG Pipeline')

BASELINE COMPARISON — CORRECTION QUALITY (F0.5)

  System                          Precision   Recall     F0.5
  ----------------------------------------------------------
  T5-GEC                             48.51%   25.19%   40.93%
  GPT-4o-mini (no RAG)                37.4%   53.73%   39.82%
  Llama-3.1-8B (no RAG)              31.17%   43.79%   33.08%

⚠️  Reminder: If you ran test runs (50 sentences),
   set MAX_SENTENCES_GPT=None and MAX_SENTENCES_LLAMA=None
   in Cells 5 and 7, then re-run cells 5-10 for full results.

📁 Output Files:
   - results/gpt4o_predictions.json
   - results/llama_predictions.json
   - results/gpt4o_per_type_scores.csv
   - results/llama_per_type_scores.csv
   - results/baseline_scores.json

➡️  Next: Run Notebook 06 — GEC-RAG Pipeline
